# 11 - Phase 3: Grad-CAM / Grad-CAM++ explainability

**Main question**: what visual regions are driving MiniConvNet's (and the strongest Phase 2
baseline's) predictions?

**No retraining. No new models.** This notebook only ever loads existing `.keras` checkpoints from
`checkpoints_local/` and runs forward+backward passes for Grad-CAM - there is no `.fit()` call
anywhere in this notebook or in `src/gradcam_utils.py`.

**Which checkpoints, decided programmatically, not assumed**: the primary model is always
MiniConvNet (`checkpoints_local/miniconvnet_single_run.keras`, from notebook 10). The secondary model
is **read from `results/fair_baseline/metrics_fair_baseline.csv`** - whichever `run_type=='finetuned'`
row has the highest accuracy - not hardcoded to VGG16. If Phase 2 is re-run later with different
results, this notebook picks up the new winner automatically.

**CPU only.** No CUDA. A single Grad-CAM pass is one forward + one backward pass per image - cheap
even on CPU - so this notebook prints a per-image time estimate for information, but does not gate on
a stop-and-ask threshold the way the training notebooks do.

**Nothing in this notebook has been executed.** It was written, not run - see the addendum at the top
of the corresponding task for why (checkpoints only exist on the machine that will actually run this).

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import numpy as np
import pandas as pd

from src.config import *
from src.data_utils import resolve_data_root, load_split, make_split_datasets

from src.gradcam_utils import (
    ensure_gradcam_dirs, GRADCAM_DIR, CHECKPOINTS_LOCAL, FAIR_METRICS_CSV,
    HIGH_CONFIDENCE_THRESHOLD, N_PER_CLASS_CATEGORY, SELECTION_CATEGORIES,
    identify_strongest_finetuned_baseline, verify_checkpoint_exists, verify_required_checkpoints,
    find_last_spatial_layer, build_gradcam_submodel,
    compute_gradcam, compute_gradcam_plusplus, check_method_reliability,
    load_image_for_gradcam, colourise_heatmap, make_overlay, border_energy_fraction,
    categorise_predictions, select_representative_samples, attach_filepaths,
    scan_for_localization_masks,
    save_gradcam_record, append_gradcam_records, load_gradcam_records, build_category_grid,
    print_gradcam_time_estimate,
)

pd.set_option('display.width', 200)
print('gradcam output dirs:', ensure_gradcam_dirs())

## Step 0a - identify the secondary model programmatically

Reads `results/fair_baseline/metrics_fair_baseline.csv` and takes the highest-accuracy
`run_type == 'finetuned'` row. **This is not assumed to be VGG16** - print whatever this cell finds.

In [ ]:
secondary = identify_strongest_finetuned_baseline()

if secondary['available']:
    print('Strongest fine-tuned baseline from Phase 2:')
    print(f"  model         : {secondary['model']}")
    print(f"  run_name      : {secondary['run_name']}")
    print(f"  accuracy      : {secondary['accuracy']:.4f}")
    print(f"  checkpoint    : {secondary['checkpoint_path']}")
    print()
    print('all finetuned rows considered, ranked:')
    for r in secondary['all_finetuned_rows']:
        print(f"  {r['run_name']:30s} {r['model']:20s} acc={r['accuracy']:.4f}")
else:
    print('Could not determine a secondary model:')
    print(' ', secondary['reason'])
    print('Run notebook 09 (Phase 2) first. Sections below that need the secondary model will')
    print('be skipped; MiniConvNet-only analysis can still proceed once its checkpoint is verified.')

PRIMARY_CHECKPOINT = str(CHECKPOINTS_LOCAL / 'miniconvnet_single_run.keras')
SECONDARY_CHECKPOINT = secondary['checkpoint_path'] if secondary['available'] else None
SECONDARY_MODEL_NAME = secondary['model'] if secondary['available'] else None

## Step 0b - verify both checkpoint files actually exist, BEFORE anything else runs

Per the addendum: if a checkpoint is missing, this notebook does **not** retrain a substitute. It
prints exactly which file is missing and the corresponding section is skipped (guarded by the
`*_OK` flags below), with a clear pointer to restore it from the manual backup made after Phase 2 /
Phase 2's checkpoint-production notebook (10).

In [ ]:
ck_status = verify_required_checkpoints(PRIMARY_CHECKPOINT, SECONDARY_CHECKPOINT,
                                        secondary_label=f'{SECONDARY_MODEL_NAME or "secondary"} '
                                                        '(strongest Phase 2 baseline)')

print(ck_status['primary']['message'])
print(ck_status['secondary']['message'])
print()

PRIMARY_OK = ck_status['primary']['exists']
SECONDARY_OK = ck_status['secondary']['exists']

if not PRIMARY_OK:
    print('=' * 78)
    print('STOP: MiniConvNet checkpoint is missing.')
    print('Restore checkpoints_local/miniconvnet_single_run.keras from backup before running')
    print('any MiniConvNet Grad-CAM cell below. No training will be started to produce one.')
    print('=' * 78)
if not SECONDARY_OK:
    print('=' * 78)
    print('NOTE: secondary-model checkpoint is missing or undetermined.')
    print('Secondary-model sections will be skipped. MiniConvNet-only analysis can still proceed')
    print('once PRIMARY_OK is True. No training will be started to produce a substitute.')
    print('=' * 78)

print()
print('PRIMARY_OK  :', PRIMARY_OK)
print('SECONDARY_OK:', SECONDARY_OK)

## Data (existing split, unchanged)

Uses the project's existing `faithful` split test set - the same one every existing result in this
project was evaluated on. No new split, no new preprocessing.

In [ ]:
sdf = load_split('faithful')
_, _, test_ds_onehot, frames = make_split_datasets(sdf, one_hot=True)     # MiniConvNet target format
_, _, test_ds_sparse, _ = make_split_datasets(sdf, one_hot=False)         # baseline target format
test_frame = frames['test'].reset_index(drop=True)
print(f'test set: {len(test_frame)} images')
print(test_frame['class'].value_counts().reindex(CLASS_NAMES).to_string())

## Step 0c - load checkpoints and identify each model's target conv layer

`find_last_spatial_layer()` recurses into any nested backbone model, so the SAME code finds the
right layer for MiniConvNet (no nesting) and for the baseline (ImageNet backbone nested as a single
layer) without a single hardcoded layer name.

**Read the printed layer name and shape before trusting anything downstream** - that is the sanity
check the addendum asks for.

In [ ]:
import tensorflow as tf

models = {}

if PRIMARY_OK:
    print('Loading MiniConvNet ...')
    mini_model = tf.keras.models.load_model(PRIMARY_CHECKPOINT)
    mini_layer_name, mini_layer_tensor, mini_layer_shape = find_last_spatial_layer(mini_model)
    print(f'  target layer: {mini_layer_name}   output shape: {mini_layer_shape}')
    try:
        mini_grad_model = build_gradcam_submodel(mini_model, mini_layer_tensor)
        models['MiniConvNet'] = {
            'model': mini_model, 'grad_model': mini_grad_model,
            'layer_name': mini_layer_name, 'layer_shape': mini_layer_shape,
            'test_ds_kind': 'onehot', 'gradcam_status': 'not yet tested',
            'gradcam_plusplus_status': 'not yet tested',
        }
        print('  Grad-CAM submodel built OK.')
    except Exception as exc:
        print(f'  !!! Could not build a Grad-CAM submodel for MiniConvNet: {exc}')
        print('  MiniConvNet Grad-CAM will be documented as UNAVAILABLE, not forced.')
        models['MiniConvNet'] = {'model': mini_model, 'grad_model': None,
                                 'layer_name': mini_layer_name, 'layer_shape': mini_layer_shape,
                                 'gradcam_status': f'unavailable: {exc}',
                                 'gradcam_plusplus_status': f'unavailable: {exc}'}
else:
    print('Skipping MiniConvNet load - checkpoint missing (see Step 0b).')

if SECONDARY_OK:
    print(f'Loading {SECONDARY_MODEL_NAME} ...')
    sec_model = tf.keras.models.load_model(SECONDARY_CHECKPOINT)
    sec_layer_name, sec_layer_tensor, sec_layer_shape = find_last_spatial_layer(sec_model)
    print(f'  target layer: {sec_layer_name}   output shape: {sec_layer_shape}')
    try:
        sec_grad_model = build_gradcam_submodel(sec_model, sec_layer_tensor)
        models[SECONDARY_MODEL_NAME] = {
            'model': sec_model, 'grad_model': sec_grad_model,
            'layer_name': sec_layer_name, 'layer_shape': sec_layer_shape,
            'test_ds_kind': 'sparse', 'gradcam_status': 'not yet tested',
            'gradcam_plusplus_status': 'not yet tested',
        }
        print('  Grad-CAM submodel built OK.')
    except Exception as exc:
        print(f'  !!! Could not build a Grad-CAM submodel for {SECONDARY_MODEL_NAME}: {exc}')
        print('  This is exactly the documented-limitation path from Step 3: a nested backbone')
        print('  layer that is not reachable from the outer graph. Not forced further.')
        models[SECONDARY_MODEL_NAME] = {'model': sec_model, 'grad_model': None,
                                        'layer_name': sec_layer_name, 'layer_shape': sec_layer_shape,
                                        'gradcam_status': f'unavailable: {exc}',
                                        'gradcam_plusplus_status': f'unavailable: {exc}'}
else:
    print(f'Skipping secondary-model load - checkpoint missing or undetermined (see Step 0b).')

## STEP 1 - select representative images: correct, incorrect, high- and normal-confidence

Run inference **once** over the whole test set for each available model (this is the only place this
notebook runs a model over the full test set), then select from that - reproducibly seeded, spanning
every class and every one of the four categories:

* correct, normal confidence
* correct, high confidence (>= 0.75)
* incorrect, normal confidence
* incorrect, high confidence (>= 0.75)

**Every category is attempted for every class. Nothing is cherry-picked to only successful examples.**
The coverage table printed below states plainly which (class, category) combinations had zero
available candidates - that gap is reported, not hidden.

In [ ]:
from src.evaluate_utils import predict as eu_predict

selections = {}
for model_name, info in models.items():
    if info.get('model') is None:
        continue
    ds = test_ds_onehot if info.get('test_ds_kind') == 'onehot' else test_ds_sparse
    y_true, y_pred, y_prob = eu_predict(info['model'], ds)
    categorised = categorise_predictions(y_true, y_pred, y_prob)
    sel = select_representative_samples(categorised, n_per_class_category=N_PER_CLASS_CATEGORY, seed=SEED)
    selected = attach_filepaths(sel['selected'], test_frame)

    print(f'--- {model_name} ---')
    print(f"selected {len(selected)} images (target up to "
          f"{len(CLASS_NAMES) * len(SELECTION_CATEGORIES) * N_PER_CLASS_CATEGORY})")
    missing = sel['coverage'][sel['coverage']['n_available'] == 0]
    if len(missing):
        print(f'  {len(missing)} (class, category) combination(s) had ZERO candidates:')
        print('   ', missing[['class', 'category']].to_string(index=False).replace(chr(10), chr(10)+'    '))
    else:
        print('  full coverage: every class x category combination had at least one candidate.')

    selections[model_name] = {'selected': selected, 'coverage': sel['coverage'],
                              'categorised': categorised}

## STEP 2 + STEP 3 - Grad-CAM and Grad-CAM++ over every selected image

For each model, for each selected image: compute Grad-CAM, then attempt Grad-CAM++. **Grad-CAM++ is
not forced** - if it raises on the first attempt for a model (NaN/Inf gradient, zero heatmap, or a
graph-connectivity failure), that model's Grad-CAM++ status is recorded as unreliable and no further
Grad-CAM++ output is produced for it; Grad-CAM output continues regardless, since the two methods are
independent.

Every image writes three PNGs (original / heatmap / overlay) plus one row to
`results/gradcam/gradcam_records.csv`.

In [ ]:
import time as _time

all_records = []

for model_name, info in models.items():
    if info.get('grad_model') is None:
        print(f'--- {model_name}: SKIPPED (no usable Grad-CAM submodel - see Step 0c) ---')
        continue

    sel_df = selections[model_name]['selected']
    n_total = len(sel_df)
    print(f'--- {model_name}: {n_total} images ---')

    gradcam_ok_count, gradcampp_ok_count = 0, 0
    gradcampp_abandoned = False
    t_start = _time.time()

    for i, (_, row) in enumerate(sel_df.reset_index(drop=True).iterrows(), start=1):
        record_id = f'{row["true_class"]}_{row["category"]}_{i:03d}'
        batch, display = load_image_for_gradcam(row['filepath'])

        # Grad-CAM
        try:
            heatmap, used_class = compute_gradcam(info['grad_model'], batch,
                                                   class_index=int(row['pred_label']))
            check = check_method_reliability(heatmap)
            if not check['reliable']:
                raise RuntimeError(f'degenerate heatmap: {check}')
            overlay = make_overlay(display, heatmap)
            rec = save_gradcam_record(record_id, model_name, 'gradcam', row, heatmap, overlay,
                                      display, info['layer_name'], status='ok')
            all_records.append(rec)
            gradcam_ok_count += 1
        except Exception as exc:
            print(f'  [gradcam] FAILED on {record_id}: {exc}')
            all_records.append({**{c: None for c in
                                   ['confidence','border_energy_fraction']},
                                'record_id': record_id, 'model': model_name, 'method': 'gradcam',
                                'filepath': row['filepath'], 'filename': row['filename'],
                                'true_class': row['true_class'], 'pred_class': row['pred_class'],
                                'confidence': float(row['confidence']), 'correct': bool(row['correct']),
                                'category': row['category'], 'target_layer': info['layer_name'],
                                'original_path': None, 'heatmap_path': None, 'overlay_path': None,
                                'status': f'failed: {exc}',
                                'timestamp': pd.Timestamp.now().isoformat()})

        # Grad-CAM++ (abandoned for this model after the first failure, per Step 3)
        if not gradcampp_abandoned:
            try:
                heatmap_pp, _ = compute_gradcam_plusplus(info['grad_model'], batch,
                                                          class_index=int(row['pred_label']))
                check_pp = check_method_reliability(heatmap_pp)
                if not check_pp['reliable']:
                    raise RuntimeError(f'degenerate heatmap: {check_pp}')
                overlay_pp = make_overlay(display, heatmap_pp)
                rec_pp = save_gradcam_record(record_id, model_name, 'gradcam_plusplus', row,
                                             heatmap_pp, overlay_pp, display, info['layer_name'],
                                             status='ok')
                all_records.append(rec_pp)
                gradcampp_ok_count += 1
            except Exception as exc:
                print(f'  [gradcam++] UNRELIABLE for {model_name} at {record_id}: {exc}')
                print(f'  Grad-CAM++ will not be attempted further for {model_name} this run - '
                      'documented as a limitation, not forced.')
                gradcampp_abandoned = True

        if i <= 2:
            print_gradcam_time_estimate(_time.time() - t_start, i, n_total, label=model_name)

    info['gradcam_status'] = (f'ok - {gradcam_ok_count}/{n_total} images succeeded' if gradcam_ok_count
                              else 'unavailable - no image succeeded')
    info['gradcam_plusplus_status'] = (
        f'ok - {gradcampp_ok_count}/{n_total} images succeeded' if gradcampp_ok_count and not gradcampp_abandoned
        else (f'partial - {gradcampp_ok_count}/{n_total} succeeded before being abandoned as unreliable'
              if gradcampp_ok_count else 'unavailable - unreliable on first attempt, not forced'))
    print(f"  {model_name} Grad-CAM      : {info['gradcam_status']}")
    print(f"  {model_name} Grad-CAM++    : {info['gradcam_plusplus_status']}")

records_path = append_gradcam_records(all_records)
print()
print('records written to', records_path)

## STEP 6 (recap) - visualisation grids per model / method / category

In [ ]:
records_df = load_gradcam_records()
print(f'{len(records_df)} total records on disk')

grid_paths = []
for model_name in models:
    for method in ('gradcam', 'gradcam_plusplus'):
        for category in SELECTION_CATEGORIES:
            p = build_category_grid(records_df, model_name, method, category, show=False)
            if p:
                grid_paths.append(p)
print(f'{len(grid_paths)} grids written to results/gradcam/grids/')
for p in grid_paths:
    print(' ', p)

## STEP 4 - what regions do the activations concentrate near?

**Language discipline enforced throughout this section**: activations "appear associated with" or
"concentrate near" a region - never "is a tumour" or "confirms the lesion location". No ground-truth
tumour localisation exists for this dataset (confirmed in Step 7 below), so nothing here can be a
confirmed clinical finding.

The border-energy-fraction column (computed in `save_gradcam_record` via
`gradcam_utils.border_energy_fraction`) is a **structural shortcut-learning diagnostic only** - the
fraction of heatmap activation mass sitting in the outer ~12% border ring of the image. It says
nothing about tumours; a high value means "attention concentrates near the image edge", which is
exactly the pattern a scanner-artifact or border shortcut would produce.

In [ ]:
ok_records = records_df[records_df['status'] == 'ok'].copy()
if len(ok_records):
    print('border-energy-fraction summary by model/method (higher = more edge-concentrated,')
    print('a possible shortcut-learning signal - NOT a tumour-localisation statement):')
    print(ok_records.groupby(['model', 'method'])['border_energy_fraction']
                    .agg(['mean', 'median', 'max', 'count']).round(4).to_string())
    print()
    high_border = ok_records[ok_records['border_energy_fraction'] > 0.5].sort_values(
        'border_energy_fraction', ascending=False)
    if len(high_border):
        print(f'{len(high_border)} record(s) with >50% of activation mass in the border ring - '
              'inspect these overlays directly (see paths below) before drawing any conclusion:')
        print(high_border[['record_id', 'model', 'method', 'border_energy_fraction',
                           'true_class', 'pred_class', 'correct',
                           'overlay_path']].to_string(index=False))
    else:
        print('No record exceeded 50% border-ring energy - activations do not appear predominantly')
        print('edge-concentrated for the images sampled here.')
else:
    print('no successful records to analyse yet')

## STEP 5 - dedicated section: high-confidence INCORRECT predictions

These are the most diagnostically important cases: the model was confidently wrong. Examined
separately and explicitly, not folded into the general grid review.

In [ ]:
hc_wrong = ok_records[ok_records['category'] == 'incorrect_high_confidence']
print(f'{len(hc_wrong)} high-confidence-incorrect record(s) across all models/methods')
if len(hc_wrong):
    print(hc_wrong[['model', 'method', 'record_id', 'true_class', 'pred_class', 'confidence',
                    'border_energy_fraction', 'overlay_path']].to_string(index=False))
    print()
    print('For each of these: open the overlay image and note, in your own words, whether the')
    print('highlighted region is (a) plausibly relevant tissue that a human could also confuse')
    print('between these two classes, (b) a region unrelated to the tissue at all (border/text/')
    print('artifact - a shortcut-learning red flag), or (c) diffuse/uninformative. Write that')
    print('assessment here once the images have actually been generated and viewed - do not')
    print('assert an interpretation sight-unseen.')
else:
    print('No high-confidence-incorrect examples were available for the sampled images - this may')
    print('mean none exist in the categories/classes sampled, not that the model has none overall.')
    print('Increase N_PER_CLASS_CATEGORY in src/gradcam_utils.py and re-run to sample more broadly')
    print('if a fuller picture of this failure mode is needed.')

## STEP 7 - is quantitative localisation evaluation possible? (checked, not assumed)

Scans the dataset directory for anything resembling a segmentation mask, bounding box, or annotation
file. **If none exist - which is the expected and, for this Kaggle CT dataset, actual answer - no
localisation metric (IoU or otherwise) is computed or approximated anywhere in this notebook.** This
cell states that plainly rather than inventing a substitute.

In [ ]:
mask_scan = scan_for_localization_masks(resolve_data_root())
print(f"localisation masks found: {mask_scan['n_mask_like_files_found']}")
print()
print(mask_scan['verdict'])

## STEP 8 - interpretation

Fill in once the cells above have actually been run and the saved images inspected. The structure
below is fixed; the content must come from what is actually observed, not from what would be
convenient to conclude.

**1. What regions does MiniConvNet appear to use?**
*(inspect `results/gradcam/grids/MiniConvNet_gradcam_*.png` and write here)*

**2. Are there possible shortcut-learning concerns (borders / text / scanner artifacts)?**
*(refer to the border-energy-fraction table in Step 4; note any record above the 0.5 flag)*

**3. What happens specifically on wrong predictions?**
*(refer to Step 5's high-confidence-incorrect section)*

**4. Does the model appear visually plausible?**
*(a qualitative judgement, phrased as "activations appear associated with X" - never a clinical claim)*

**5. What can and cannot be concluded from this analysis alone?**
*(at minimum: no ground-truth localisation exists for this dataset - Step 7 - so nothing here confirms*
*or refutes anatomical correctness; Grad-CAM/Grad-CAM++ show what the model responds to, not what is*
*clinically true; the sample is small and seeded, not exhaustive)*

In [ ]:
gradcam_statuses = {m: info.get('gradcam_status', 'not run') for m, info in models.items()}
gradcampp_statuses = {m: info.get('gradcam_plusplus_status', 'not run') for m, info in models.items()}
n_images_total = sum(len(s['selected']) for s in selections.values()) if selections else 0

print('PHASE:                Phase 3 - Grad-CAM / Grad-CAM++ explainability')
print('MODELS:                MiniConvNet (primary)'
      + (f', {SECONDARY_MODEL_NAME} (secondary, strongest Phase 2 fine-tuned baseline)'
         if SECONDARY_MODEL_NAME else ' | secondary: NOT DETERMINED (Phase 2 not run / no finetuned rows)'))
print(f'NUMBER OF IMAGES:      {n_images_total} selected (see per-model coverage tables in Step 1)')
print('GRAD-CAM STATUS:')
for m, s in gradcam_statuses.items():
    print(f'    {m:20s} {s}')
print('GRAD-CAM++ STATUS:')
for m, s in gradcampp_statuses.items():
    print(f'    {m:20s} {s}')
print('KEY OBSERVATIONS:      fill in from Step 8 above once run')
print('FILES CREATED:')
print(f'    {GRADCAM_DIR}/gradcam_records.csv')
print(f'    {GRADCAM_DIR}/images/  (original/heatmap/overlay PNGs per record)')
print(f'    {GRADCAM_DIR}/grids/   ({len(grid_paths) if "grid_paths" in dir() else 0} category grids)')
print('CPU TIME:              sum of the per-image timing prints above (informational only, no gate)')
print('LIMITATIONS:           no ground-truth localisation masks exist for this dataset (Step 7),')
print('                       so no IoU/localisation metric is computed anywhere; sample is small')
print('                       and seeded (reproducible, not exhaustive); border-energy-fraction is a')
print('                       structural shortcut-learning diagnostic, not a clinical measurement;')
print('                       Grad-CAM++ reliability is architecture-dependent and was tested, not')
print('                       assumed, per model (see the GRAD-CAM++ STATUS lines above).')